# Exploratory Data Analysis: Insurance Risk Analytics

## Objective
Perform comprehensive EDA on 18 months of historical insurance claim data (Feb 2014 – Aug 2015) to:
- Understand data quality and structure
- Identify key risk drivers
- Discover patterns in claim frequency and severity
- Inform hypothesis testing and modeling strategies

## Key Metrics
- **Loss Ratio** = TotalClaims / TotalPremium
- **Margin** = TotalPremium − TotalClaims
- **Claim Frequency** = proportion of policies with claims
- **Claim Severity** = average claim amount (given claim occurred)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

from src.data_loader import DataLoader
from src.eda_utils import EDAAnalyzer, get_loss_ratio_by_group, get_claim_frequency

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries imported successfully')

## 1. Data Loading & Validation

In [ ]:
# Load data
data_path = '../data/insurance_data.csv'
loader = DataLoader(data_path)

try:
    df = loader.load_data()
    print(f'Data loaded successfully: {df.shape}')
    print(f'\nFirst few rows:')
    print(df.head())
except FileNotFoundError:
    print(f'Data file not found at {data_path}')
    print('Please ensure the insurance_data.csv file is in the data/ directory')
    df = None

In [ ]:
if df is not None:
    # Data validation report
    validation_report = loader.validate_data()
    print('Data Validation Report:')
    print(f'Shape: {validation_report["shape"]}')
    print(f'Total cells: {validation_report["total_cells"]}')
    print(f'Missing cells: {validation_report["missing_cells"]}')
    print(f'Missing percentage: {validation_report["missing_percentage"]:.2f}%')
    print(f'\nData types:')
    for col, dtype in validation_report['dtypes'].items():
        print(f'  {col}: {dtype}')

## 2. Data Quality Assessment

In [ ]:
if df is not None:
    analyzer = EDAAnalyzer(df)
    
    # Quality report
    quality_report = analyzer.data_quality_report()
    print('Data Quality Report:')
    for key, value in quality_report.items():
        print(f'{key}: {value}')
    
    # Missing values analysis
    print('\nMissing Values Analysis:')
    missing_df = analyzer.missing_values_analysis()
    print(missing_df[missing_df['Missing_Count'] > 0])

In [ ]:
if df is not None:
    # Visualize missing values
    analyzer.plot_missing_values()

## 3. Univariate Analysis

In [ ]:
if df is not None:
    # Summary statistics
    print('Summary Statistics:')
    print(loader.get_summary_statistics(df))

In [ ]:
if df is not None:
    # Plot numerical distributions
    numerical_cols = ['TotalPremium', 'TotalClaims', 'SumInsured', 'CustomValueEstimate']
    available_cols = [col for col in numerical_cols if col in df.columns]
    analyzer.plot_numerical_distributions(available_cols)

In [ ]:
if df is not None:
    # Plot categorical distributions
    categorical_cols = ['Province', 'VehicleType', 'Gender', 'CoverType']
    available_cols = [col for col in categorical_cols if col in df.columns]
    analyzer.plot_categorical_distributions(available_cols, top_n=10)

## 4. Outlier Detection

In [ ]:
if df is not None:
    # Boxplots for outlier detection
    numerical_cols = ['TotalPremium', 'TotalClaims', 'CustomValueEstimate']
    available_cols = [col for col in numerical_cols if col in df.columns]
    analyzer.plot_boxplots(available_cols)

In [ ]:
if df is not None:
    # Identify outliers in TotalClaims
    if 'TotalClaims' in df.columns:
        outliers = analyzer.get_outliers('TotalClaims', method='iqr')
        print(f'Number of outliers in TotalClaims: {len(outliers)}')
        print(f'Percentage of data: {len(outliers)/len(df)*100:.2f}%')
        print(f'\nOutlier statistics:')
        print(outliers['TotalClaims'].describe())

## 5. Derived Metrics & Key Insights

In [ ]:
if df is not None:
    # Create derived metrics
    df_analysis = loader.create_derived_metrics(df)
    
    # Overall portfolio metrics
    total_premium = df_analysis['TotalPremium'].sum()
    total_claims = df_analysis['TotalClaims'].sum()
    overall_loss_ratio = total_claims / total_premium
    overall_margin = total_premium - total_claims
    claim_frequency = df_analysis['ClaimIndicator'].mean()
    
    print('Overall Portfolio Metrics:')
    print(f'Total Premium: R{total_premium:,.2f}')
    print(f'Total Claims: R{total_claims:,.2f}')
    print(f'Loss Ratio: {overall_loss_ratio:.2%}')
    print(f'Margin: R{overall_margin:,.2f}')
    print(f'Claim Frequency: {claim_frequency:.2%}')
    print(f'Average Claim (when claim occurs): R{df_analysis[df_analysis["TotalClaims"] > 0]["TotalClaims"].mean():,.2f}')

## 6. Bivariate & Multivariate Analysis

In [ ]:
if df is not None:
    # Correlation matrix
    numerical_cols = df_analysis.select_dtypes(include=[np.number]).columns
    analyzer.plot_correlation_matrix(numerical_cols.tolist())

In [ ]:
if df is not None and 'TotalPremium' in df.columns and 'TotalClaims' in df.columns:
    # Scatter plot: Premium vs Claims
    analyzer.plot_scatter('TotalPremium', 'TotalClaims')

## 7. Geographic Trends

In [ ]:
if df is not None and 'Province' in df.columns:
    # Loss ratio by province
    loss_by_province = get_loss_ratio_by_group(df_analysis, 'Province')
    print('Loss Ratio by Province:')
    print(loss_by_province)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    loss_by_province.plot(x='Province', y='LossRatio', kind='bar', ax=axes[0], color='coral')
    axes[0].set_title('Loss Ratio by Province')
    axes[0].set_ylabel('Loss Ratio')
    axes[0].set_xlabel('Province')
    
    loss_by_province.plot(x='Province', y='Margin', kind='bar', ax=axes[1], color='lightgreen')
    axes[1].set_title('Margin by Province')
    axes[1].set_ylabel('Margin (R)')
    axes[1].set_xlabel('Province')
    
    loss_by_province.plot(x='Province', y='PolicyCount', kind='bar', ax=axes[2], color='skyblue')
    axes[2].set_title('Policy Count by Province')
    axes[2].set_ylabel('Number of Policies')
    axes[2].set_xlabel('Province')
    
    plt.tight_layout()
    plt.show()

## 8. Vehicle Analysis

In [ ]:
if df is not None and 'VehicleType' in df.columns:
    # Loss ratio by vehicle type
    loss_by_vehicle = get_loss_ratio_by_group(df_analysis, 'VehicleType')
    print('Loss Ratio by Vehicle Type:')
    print(loss_by_vehicle)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    loss_by_vehicle.plot(x='VehicleType', y='LossRatio', kind='bar', ax=axes[0], color='coral')
    axes[0].set_title('Loss Ratio by Vehicle Type')
    axes[0].set_ylabel('Loss Ratio')
    axes[0].set_xlabel('Vehicle Type')
    
    loss_by_vehicle.plot(x='VehicleType', y='PolicyCount', kind='bar', ax=axes[1], color='skyblue')
    axes[1].set_title('Policy Count by Vehicle Type')
    axes[1].set_ylabel('Number of Policies')
    axes[1].set_xlabel('Vehicle Type')
    
    plt.tight_layout()
    plt.show()

## 9. Gender Analysis

In [ ]:
if df is not None and 'Gender' in df.columns:
    # Loss ratio by gender
    loss_by_gender = get_loss_ratio_by_group(df_analysis, 'Gender')
    print('Loss Ratio by Gender:')
    print(loss_by_gender)
    
    # Claim frequency by gender
    freq_by_gender = get_claim_frequency(df_analysis, 'Gender')
    print('\nClaim Frequency by Gender:')
    print(freq_by_gender)
    
    # Visualize
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    loss_by_gender.plot(x='Gender', y='LossRatio', kind='bar', ax=axes[0], color='coral')
    axes[0].set_title('Loss Ratio by Gender')
    axes[0].set_ylabel('Loss Ratio')
    axes[0].set_xlabel('Gender')
    
    freq_by_gender.plot(x='Gender', y='ClaimFrequency', kind='bar', ax=axes[1], color='lightgreen')
    axes[1].set_title('Claim Frequency by Gender')
    axes[1].set_ylabel('Claim Frequency')
    axes[1].set_xlabel('Gender')
    
    plt.tight_layout()
    plt.show()

## 10. Key Findings Summary

In [ ]:
if df is not None:
    print('KEY FINDINGS FROM EDA')
    print('='*60)
    print(f'\n1. PORTFOLIO OVERVIEW')
    print(f'   - Total Policies: {len(df_analysis):,}')
    print(f'   - Total Premium: R{total_premium:,.2f}')
    print(f'   - Total Claims: R{total_claims:,.2f}')
    print(f'   - Overall Loss Ratio: {overall_loss_ratio:.2%}')
    print(f'   - Overall Margin: R{overall_margin:,.2f}')
    
    print(f'\n2. CLAIM PATTERNS')
    print(f'   - Claim Frequency: {claim_frequency:.2%}')
    print(f'   - Policies with Claims: {df_analysis["ClaimIndicator"].sum():,}')
    print(f'   - Avg Claim Amount (when claim occurs): R{df_analysis[df_analysis["TotalClaims"] > 0]["TotalClaims"].mean():,.2f}')
    
    if 'Province' in df.columns:
        print(f'\n3. GEOGRAPHIC INSIGHTS')
        highest_loss_prov = loss_by_province.iloc[0]
        lowest_loss_prov = loss_by_province.iloc[-1]
        print(f'   - Highest Loss Ratio: {highest_loss_prov["Province"]} ({highest_loss_prov["LossRatio"]:.2%})')
        print(f'   - Lowest Loss Ratio: {lowest_loss_prov["Province"]} ({lowest_loss_prov["LossRatio"]:.2%})')
    
    if 'VehicleType' in df.columns:
        print(f'\n4. VEHICLE INSIGHTS')
        highest_loss_veh = loss_by_vehicle.iloc[0]
        lowest_loss_veh = loss_by_vehicle.iloc[-1]
        print(f'   - Highest Loss Ratio: {highest_loss_veh["VehicleType"]} ({highest_loss_veh["LossRatio"]:.2%})')
        print(f'   - Lowest Loss Ratio: {lowest_loss_veh["VehicleType"]} ({lowest_loss_veh["LossRatio"]:.2%})')
    
    if 'Gender' in df.columns:
        print(f'\n5. GENDER INSIGHTS')
        for _, row in loss_by_gender.iterrows():
            print(f'   - {row["Gender"]}: Loss Ratio {row["LossRatio"]:.2%}, Margin R{row["Margin"]:,.2f}')
    
    print(f'\n6. DATA QUALITY')
    print(f'   - Missing Data: {quality_report["missing_percentage"]:.2f}%')
    print(f'   - Duplicate Rows: {quality_report["duplicate_rows"]}')
    print(f'   - Numerical Features: {quality_report["columns_by_type"]["numerical"]}')
    print(f'   - Categorical Features: {quality_report["columns_by_type"]["categorical"]}')

## 11. Recommendations for Next Steps

Based on the EDA findings:

1. **Hypothesis Testing**: Validate observed differences in loss ratios across provinces, vehicle types, and gender using statistical tests
2. **Feature Engineering**: Create derived features (vehicle age, policy duration, etc.) for modeling
3. **Segmentation**: Identify low-risk segments for targeted marketing
4. **Modeling**: Build predictive models for claim severity and probability
5. **Pricing Strategy**: Develop risk-based premium calculation framework